# ASR Transcription with Paraformer-zh

Load WAV files from a folder, run Paraformer-zh (via FunASR) with FSMN-VAD,
and save word-level ASR results to JSONL.

In [ ]:
# %pip install funasr modelscope soundfile

In [ ]:
from pathlib import Path

# -- Configure these paths --
WAV_DIR = Path("../data/wav")       # folder containing .wav files
OUTPUT_PATH = Path("../data/asr_results.jsonl")

WAV_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
from funasr import AutoModel

model = AutoModel(
    model="paraformer-zh",   # ASR — best Chinese accuracy at 220M params
    vad_model="fsmn-vad",    # VAD with timestamps
    punc_model="ct-punc",    # punctuation restoration
    device="mps",            # MPS for Mac
)
print("Model loaded.")

In [ ]:
wav_files = sorted(WAV_DIR.glob("*.wav"))
print(f"Found {len(wav_files)} WAV files in {WAV_DIR}")
for f in wav_files[:5]:
    print(f"  {f.name}")
if len(wav_files) > 5:
    print(f"  ... and {len(wav_files) - 5} more")

In [ ]:
import json

for i, wav_path in enumerate(wav_files):
    res = model.generate(
        input=str(wav_path),
        return_raw_text=True,
        sentence_timestamp=True,
    )

    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        for item in res:
            record = {
                "file": wav_path.name,
                "text": item.get("text", ""),
                "sentences": item.get("sentence_info", []),
                "timestamp": item.get("timestamp", []),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    if (i + 1) % 10 == 0 or (i + 1) == len(wav_files):
        print(f"[{i + 1}/{len(wav_files)}] {wav_path.name}")

print(f"\nDone. Results saved to {OUTPUT_PATH}")

## Speed

With CPU

> rtf_avg: 0.069, time_speech:  2068.000, time_escape: 143.4924.00s/it]

With MPS

> rtf_avg: 0.009, time_speech:  2068.000, time_escape: 19.374:9.86s/it]


In [ ]:
# Preview results
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        print(f"--- {r['file']} ---")
        print(f"  text: {r['text']}")
        if r["sentences"]:
            print(f"  sentences: {r['sentences'][:3]}")
        print()